# MTAM reproduction — ZuCo sentiment analysis

This is the single Colab notebook for the `reproduction` branch of `parmisbathaeiyan/EEG_Language_Alignment`. Data and run outputs remain in Google Drive.

**Current step: v16, a bounded feature-construction audit.** It inspects only the precomputed sentence and word/fixation EEG frequency-band fields relevant to the paper and released repository. It does not train a model, inspect raw voltage, choose a split, or invent new EEG features.

The audit asks whether the stored word-level band arrays can reconstruct each participant's stored sentence-level `mean_t1` through `mean_g2` arrays under a small set of declared pooling rules. It also records field shapes, coverage, and missingness. This is the final data-construction check before the frozen multi-seed MLP-EEG reproduction.

No GPU or additional package installation is required for v16.

## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OG_ZUCO_SR_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'
RESULTS_ROOT = '/content/drive/MyDrive/Thesis/Results/reproduce_EEG_Language_Alignment'
FORK_URL = 'https://github.com/parmisbathaeiyan/EEG_Language_Alignment.git'
BRANCH = 'reproduction'

import os
assert os.path.isdir(OG_ZUCO_SR_DIR), f'SR .mat folder not found: {OG_ZUCO_SR_DIR}'
mat_files = sorted(name for name in os.listdir(OG_ZUCO_SR_DIR) if name.endswith('.mat'))
assert mat_files, f'No .mat files found in {OG_ZUCO_SR_DIR}'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print('Drive mounted. SR files:', mat_files)

## 2. Clone the current reproduction branch

Re-running this cell refreshes the Colab copy from GitHub. The Drive data is not touched.

In [ ]:
%cd /content
!rm -rf /content/EEG_Language_Alignment
!git clone --branch {BRANCH} {FORK_URL}
%cd /content/EEG_Language_Alignment

print('Checked-out commit:')
!git rev-parse HEAD
print('\nChanges relative to upstream:')
!git remote add upstream https://github.com/Jason-Qiu/EEG_Language_Alignment.git 2>/dev/null; git fetch -q upstream
!git log --oneline upstream/main..{BRANCH}

## 3. Link the original ZuCo `.mat` files

The audit reads the files in place through symlinks; it does not duplicate or modify the Drive data.

In [ ]:
%cd /content/EEG_Language_Alignment
import os

os.makedirs('data/SR', exist_ok=True)
for filename in mat_files:
    destination = os.path.join('data/SR', filename)
    if not os.path.lexists(destination):
        os.symlink(os.path.join(OG_ZUCO_SR_DIR, filename), destination)

linked_files = sorted(name for name in os.listdir('data/SR') if name.endswith('.mat'))
assert linked_files == mat_files, (linked_files, mat_files)
print('Linked SR files:', linked_files)

import numpy as np, scipy
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__)

## 4. Run v16 — reproduction-scoped feature audit

This produces one timestamped log and one JSON in Drive under `v16_reproductionFeatureAudit`. For each participant file it inventories sentence and word fields, shapes, usable 104-electrode arrays, and NaN patterns. It then compares each stored sentence `mean_<band>` vector with FFD/SFD/GD/GPT/TRT word-vector pooling candidates.

A failed reconstruction does **not** mean that the data is invalid; it means only that the declared simple pooling rule does not explain the stored sentence field.

In [ ]:
import datetime, os, subprocess

audit_timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
audit_dir = os.path.join(RESULTS_ROOT, 'v16_reproductionFeatureAudit')
log_dir = os.path.join(audit_dir, 'logs')
os.makedirs(log_dir, exist_ok=True)

audit_json = os.path.join(audit_dir, f'feature_construction_audit_{audit_timestamp}.json')
audit_log = os.path.join(log_dir, f'feature_construction_audit_{audit_timestamp}.txt')
audit_command = [
    'python', '-u', 'audit_reproduction_features.py',
    '--eeg_dir', 'data/SR',
    '--output_json', audit_json,
    '--electrodes', '104',
]

print('Running v16 feature audit')
print('JSON:', audit_json)
print('Log:', audit_log)
print('-' * 60)
with open(audit_log, 'w') as log_file:
    log_file.write('Command: ' + ' '.join(audit_command) + '\n' + '-' * 60 + '\n')
    process = subprocess.Popen(
        audit_command, cwd='/content/EEG_Language_Alignment',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'v16 audit failed with exit code {return_code}. See {audit_log}')
print('-' * 60)
print('v16 complete.')

## 5. Show the compact result summary

Run this after section 4. The complete machine-readable evidence remains in the JSON.

In [ ]:
import json

with open(audit_json) as audit_file:
    audit = json.load(audit_file)

source = audit['source']
print('Commit:', audit['provenance']['git_commit'])
print('Included participant files:', len(source['included_files']))
print('Excluded files:', source['excluded_files'])
print('Sentence records:', source['total_sentence_records'])
print('Word records:', source['total_word_records'])
print('Sentence fields:', ', '.join(audit['sentence_field_inventory']))

word_band_fields = [
    name for name in audit['word_field_inventory']
    if any(name.startswith(prefix + '_') for prefix in ('FFD', 'SFD', 'GD', 'GPT', 'TRT'))
]
print('Word/fixation band fields:', ', '.join(word_band_fields) or 'none found')

print('\nBest reconstruction candidates:')
for row in audit['reconstruction']['ranked_best_first'][:12]:
    print(
        f"{row['window']:>11} | {row['pooling_rule']:<40} | {row['band']} | "
        f"pairs={row['sentence_band_vector_pairs']:>4} | "
        f"relative RMSE={row['relative_rmse']:.6g} | "
        f"correlation={row['flattened_pearson_correlation']}"
    )

print('\nSaved JSON:', audit_json)
print('Saved log:', audit_log)

---
### After v16
Tell Codex that v16 finished. We will read the Drive JSON and determine whether the supplied sentence features are explained by any declared word-level pooling rule. Regardless of whether reconstruction succeeds, we will preserve unresolved ambiguities and proceed to the frozen multi-seed MLP unless the audit exposes one objective implementation/data defect.